In [6]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="/home/ubuntu/LLM Rerankers/api_keys.env") 

True

### DeepSeek

In [7]:
import json
import pandas as pd
from pydantic import RootModel, ValidationError
from typing import List
from dotenv import load_dotenv

load_dotenv(dotenv_path="/home/ubuntu/LLM Rerankers/api_keys.env")

class RerankedIndices(RootModel[List[int]]):
    pass

In [48]:
target_num_of_chunks = 256
target_num_of_overlaps = 64
original_num_of_chunks = 1024
original_num_of_overlaps = 256
number_of_docs = 50
number_of_docs_minus_one = number_of_docs - 1

chunks_file_path = f"/home/ubuntu/RAG Pipeline/March 27 Rep Chunks/{target_num_of_chunks}-{target_num_of_overlaps}-from-{original_num_of_chunks}-{original_num_of_overlaps}-{number_of_docs}-representative_chunks.json"

with open(chunks_file_path, 'r', encoding='utf-8') as f:
    retrieved_chunks = json.load(f)

eval_csv_path = "/home/ubuntu/[March 27, 2025] Final Dataset/sampled_final_cleaned_acl_global_context_dataset_eval.csv"
df = pd.read_csv(eval_csv_path)

# Extract the 'masked_cit_context' column and convert to list
masked_contexts = df["masked_cit_context"].tolist()

retrieved_chunks = retrieved_chunks[692]
query = masked_contexts[692]

In [49]:
from langchain.prompts import PromptTemplate

q_template = (
    "Given this list of retrieved chunks (total: {number_of_docs}):\n"
    "{retrieved_chunks}\n\n"

    "And given the query:\n"
    "{query}\n\n"

    "Rerank the chunks above according to their relevance to the query. Rank them from most relevant to least relevant.\n"
    "After reranking, output the current order of their indices based on the original order. Format it as a Python list of integers. For example, [9, 7, 6, 5, 3, 1, 8, 2, 4, 0].\n"
    "There is no need to explain your answers. You can simply output the Python list of indices.\n\n"

    "Make sure that:\n"
    "- The list contains only integers between 0 and {number_of_docs_minus_one}.\n"
    "- The length of the list is exactly {number_of_docs}."
)

q_prompt = PromptTemplate(
    input_variables=["number_of_docs", "retrieved_chunks", "query", "number_of_docs_minus_one"],
    template=q_template
)

filled_prompt = q_prompt.format(
        number_of_docs=number_of_docs,
        retrieved_chunks=retrieved_chunks,
        query=query,
        number_of_docs_minus_one=number_of_docs_minus_one
    )

print(filled_prompt)

Given this list of retrieved chunks (total: 50):
['science where statistical techniques for gender attribution can be useful: e.g., they can help understanding demographic characteristics of user-created web text today, which can provide new insight to social science as well as intelligent marketing and', 'are most talked about by males and females, and what products and services are liked or disliked by men and women. Knowing this information is crucial for market intelligence because the information can be exploited in targeted advertising and also', 'implemented variations of the method given in (Argamon et al., 2003) . Here, the improvement of our techniques is even greater. Related Work There have been several recent papers on gender classification of blogs (e.g., Schler et al., 2006 , Argamon et', 'provide new insight to social science as well as intelligent marketing and', 'of the features used in our system. Recall our system includes all their features and our own POS pattern 

In [50]:
from openai import OpenAI

client = OpenAI(api_key=os.getenv("DEEPSEEK_API_KEY"), base_url="https://api.deepseek.com")

try:
        response = client.chat.completions.create(
            model="deepseek-chat",
            messages=[
                {"role": "system", "content": "You are a helpful assistant"},
                {"role": "user", "content": filled_prompt},
            ],
            stream=False,
            temperature=0.1
        )

        text_response = response.choices[0].message.content

        # Default structured response
        structured_response = None

        # Clean the text response (similar to your Gemini code)
        cleaned_response = text_response.strip()
        if cleaned_response.startswith("```") and cleaned_response.endswith("```"):
            cleaned_response = cleaned_response[3:-3].strip()
            if cleaned_response.startswith("json"):
                cleaned_response = cleaned_response[4:].strip()
            if cleaned_response.startswith("python"):
                cleaned_response = cleaned_response[6:].strip()

        # Attempt to parse the cleaned text response as JSON and validate with Pydantic
        try:
            json_output = json.loads(cleaned_response)
            structured_response = RerankedIndices.model_validate_json(json.dumps(json_output))
        except (json.JSONDecodeError, ValidationError) as e:
            print(f"Error parsing or validating JSON from DeepSeek: {e}")
            print(f"Failed to parse JSON. Raw response from DeepSeek: '{text_response}'")
            structured_response = None

        #return text_response, structured_response

except Exception as e:
    print(f"Error calling DeepSeek API: {e}")
    #return "", None

print(text_response)
print(structured_response.root)

Error calling DeepSeek API: Error code: 400 - {'error': {'message': 'Content Exists Risk', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_request_error'}}
[17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
[17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]


### Gemini

In [1]:
import os
from google import genai
from google.genai.types import GenerateContentConfig
import json
import pandas as pd
from langchain.prompts import PromptTemplate
from pydantic import RootModel, ValidationError
from typing import List
from dotenv import load_dotenv

load_dotenv(dotenv_path="/home/ubuntu/LLM Rerankers/api_keys.env")

class RerankedIndices(RootModel[List[int]]):
    pass

def generate(filled_prompt):
    client = genai.Client(
        api_key=os.getenv("GEMINI_API_KEY"),
    )

    model = "gemini-2.0-flash-thinking-exp-01-21"
    contents = filled_prompt
    
    generate_content_config = GenerateContentConfig(
        temperature=0.1,
        max_output_tokens=15000,
        response_mime_type="text/plain",  # Revert to text/plain
    )

    full_response = client.models.generate_content(
        model=model,
        contents=contents,
        config=generate_content_config,
    )

    # Default response
    text_response = "No response generated from the model."
    structured_response = None  # Initialize structured_response

    # Check if the response is valid
    if full_response and full_response.candidates and full_response.candidates[0]:

        # Extracts the candidate response
        candidate = full_response.candidates[0]

        if candidate.content and candidate.content.parts:
                # Extract text response
                text_response = ""
                for each in candidate.content.parts:
                    text_response += each.text

                # Clean the text response: remove code block markers and strip whitespace
                cleaned_response = text_response.strip()
                if cleaned_response.startswith("```") and cleaned_response.endswith("```"):
                    # Remove the outer ``` if present
                    cleaned_response = cleaned_response[3:-3].strip()
                    # Remove language identifier if present (e.g., ```json)
                    if cleaned_response.startswith("json"):
                        cleaned_response = cleaned_response[4:].strip()
                    if cleaned_response.startswith("python"):
                        cleaned_response = cleaned_response[6:].strip()

                # Attempt to parse the cleaned text response as JSON and validate with Pydantic
                try:
                    json_output = json.loads(cleaned_response)
                    structured_response = RerankedIndices.model_validate_json(json.dumps(json_output))
                except (json.JSONDecodeError, ValidationError) as e:
                    print(f"Error parsing or validating JSON: {e}")
                    print(f"Failed to parse JSON. Raw response: '{text_response}'")
                    structured_response = None

    # Return both the raw text response and the structured response
    return text_response, structured_response

if __name__ == "__main__":
    
    target_num_of_chunks = 256
    target_num_of_overlaps = 64
    original_num_of_chunks = 1024
    original_num_of_overlaps = 256
    number_of_docs = 20
    number_of_docs_minus_one = number_of_docs - 1

    chunks_file_path = f"/home/ubuntu/RAG Pipeline/March 27 Chunks/{original_num_of_chunks}-{original_num_of_overlaps}-{number_of_docs}-retrieved_chunks.json"

    with open(chunks_file_path, 'r', encoding='utf-8') as f:
        all_retrieved_chunks = json.load(f)

    print(len(all_retrieved_chunks))
    print(len(all_retrieved_chunks[0]))

    eval_csv_path = "/home/ubuntu/[March 27, 2025] Final Dataset/sampled_final_cleaned_acl_global_context_dataset_eval.csv"
    df = pd.read_csv(eval_csv_path)

    # Extract the 'masked_cit_context' column and convert to list
    masked_contexts = df["masked_cit_context"].tolist()
    print(len(masked_contexts))

    q_template = (
        "Given this list of retrieved chunks (total: {number_of_docs}):\n"
        "{retrieved_chunks}\n\n"

        "And given the query:\n"
        "{query}\n\n"

        "Rerank the chunks above according to their relevance to the query. Rank them from most relevant to least relevant.\n"
        "After reranking, output the current order of their indices based on the original order. Format it as a Python list of integers. For example, [9, 7, 6, 5, 3, 1, 8, 2, 4, 0].\n"
        "There is no need to explain your answers. You can simply output the Python list of indices.\n\n"

        "Make sure that:\n"
        "- The list contains only integers between 0 and {number_of_docs_minus_one}.\n"
        "- The length of the list is exactly {number_of_docs}."
    )

    q_prompt = PromptTemplate(
        input_variables=["number_of_docs", "retrieved_chunks", "query", "number_of_docs_minus_one"],
        template=q_template
    )

    for i in range(2):
        retrieved_chunks = all_retrieved_chunks[i]
        query = masked_contexts[i]    

        filled_prompt = q_prompt.format(
                number_of_docs=number_of_docs,
                retrieved_chunks=retrieved_chunks,
                query=query,
                number_of_docs_minus_one=number_of_docs_minus_one
            )

        text_response, structured_response = generate(filled_prompt)
        print(text_response)
        print(type(text_response))
        print(structured_response)
        print(type(structured_response))


1245
20
1245
```python
[2, 0, 1, 19, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]
```
<class 'str'>
root=[2, 0, 1, 19, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]
<class '__main__.RerankedIndices'>
[11, 1, 19, 6, 2, 3, 4, 5, 15, 7, 10, 8, 9, 13, 12, 14, 16, 17, 18, 0]
<class 'str'>
root=[11, 1, 19, 6, 2, 3, 4, 5, 15, 7, 10, 8, 9, 13, 12, 14, 16, 17, 18, 0]
<class '__main__.RerankedIndices'>


In [14]:
print(text_response[0])
print(structured_response.root)
print(type(structured_response.root))

[
[6, 7, 8, 9, 10, 16, 17, 18, 19, 11, 12, 13, 14, 15, 0, 1, 2, 3, 4, 5]
<class 'list'>


In [1]:
list_1 = [1, 0, 22, 23, 38, 3, 20, 2, 29, 24, 25, 26, 27, 28, 30, 36, 37, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 21, 31]

unexpected_indices = []
unexpected_values = []
for i in range(len(list_1)): # Iterate through the entire list to find unexpected indices
    if list_1[i] >= 35:
        unexpected_indices.append(i)
        unexpected_values.append(list_1[i])

print(f"Unexpected Indices: {unexpected_indices}")
print(f"Unexpected Values: {unexpected_values}")

set_list_1 = set(list_1)
list_2 = set(range(35))
list_3 = sorted(list(list_2 - set_list_1)) # Sort list_3 for consistent mapping
print(f"List 3 (sorted missing values): {list_3}")

# Sort both lists of unexpected values and missing values
sorted_unexpected_values = sorted(unexpected_values)
sorted_list_3 = sorted(list_3) # Ensure list_3 is sorted

# Create a mapping based on the sorted order
mapping = {}
for i in range(len(sorted_unexpected_values)):
    if i < len(sorted_list_3):
        mapping[sorted_unexpected_values[i]] = sorted_list_3[i]
    else:
        print("Warning: Not enough missing values to map all unexpected values.")
        break

print(f"Mapping: {mapping}")

# Apply the mapping to change the unexpected_values with the list_3 values at the unexpected_indices
mapped_list_1 = list(list_1) # Create a copy to modify

for i in range(len(unexpected_indices)):
    index_to_change = unexpected_indices[i]
    original_value = mapped_list_1[index_to_change]
    if original_value in mapping:
        mapped_list_1[index_to_change] = mapping[original_value]
    else:
        print(f"Warning: No mapping found for value {original_value} at index {index_to_change}.")

print(f"Original list_1: {list_1}")
print(f"Mapped list_1: {mapped_list_1}")

Unexpected Indices: [4, 15, 16]
Unexpected Values: [38, 36, 37]
List 3 (sorted missing values): [32, 33, 34]
Mapping: {36: 32, 37: 33, 38: 34}
Original list_1: [1, 0, 22, 23, 38, 3, 20, 2, 29, 24, 25, 26, 27, 28, 30, 36, 37, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 21, 31]
Mapped list_1: [1, 0, 22, 23, 34, 3, 20, 2, 29, 24, 25, 26, 27, 28, 30, 32, 33, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 21, 31]
